# Bài 4 · Làm quen với pandas

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài học** — sau notebook này, bạn sẽ:

1. Đọc CSV thành DataFrame và chạy **thói quen 5 bước** khám phá một dataset lạ.
2. Chọn cột, lọc hàng bằng mặt nạ bool / `isin` / `loc`, thêm cột mới vector hoá.
3. Trả lời câu hỏi thống kê bằng chuỗi thao tác (method chaining) ngắn.
4. Đo mức thiếu dữ liệu của từng cột (chưa xử lý — bài 10).

Dữ liệu: **18.534 phòng Airbnb ở Santiago, Chile** (snapshot 29/06/2026, bản rút gọn
`visualisations/listings.csv` của Inside Airbnb).

## 1. Nạp dữ liệu & thói quen 5 bước

In [ ]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
df = pd.read_csv(URL)

# Bước 1: bảng có bao nhiêu dòng và cột?
df.shape

In [ ]:
# Bước 2: xem một số dòng ngẫu nhiên để tránh chỉ quan sát phần đầu file
df.sample(5, random_state=1)

In [ ]:
# Bước 3: kiểu gì, thiếu đâu?
df.info()

In [ ]:
# Bước 4: cột số có gì lạ?
df["price"].describe().round()

Ba con số cần lưu ý ở `price`:

- `count` 17.688 trong khi bảng có 18.534 dòng → **846 phòng không có giá**;
- `mean` 118.200 gấp đôi `50%` (median 59.000) → phân phối **lệch phải nặng**;
- `max` 97.000.045 CLP/đêm (~2,6 tỷ VND) → cần được kiểm tra như một **ứng viên ngoại lai (outlier) hoặc lỗi nhập**.

Chưa cần xử lý trong bài này, nhưng các kết luận về giá cần xét đến ba đặc điểm trên.

In [ ]:
# Bước 5: cột phân loại gồm những nhóm nào?
df["room_type"].value_counts(normalize=True).round(3)

## 2. Chọn cột, lọc hàng

In [ ]:
# Một tên -> Series; danh sách tên -> DataFrame (2 lớp ngoặc!)
print(type(df["price"]))
print(type(df[["name", "price"]]))

In [ ]:
# Mặt nạ bool + & | ~ (ngoặc quanh từng vế) — tương tự NumPy
re_nguyen_can = df[(df["room_type"] == "Entire home/apt") & (df["price"] < 50000)]
print(len(re_nguyen_can), "căn nguyên căn giá < 50.000 CLP/đêm")

In [ ]:
# isin: lọc theo danh sách giá trị
khu_trung_tam = df[df["neighbourhood"].isin(["Santiago", "Providencia", "Las Condes"])]
khu_trung_tam["neighbourhood"].value_counts()

In [ ]:
# loc[chọn_hàng, chọn_cột] — lọc và cắt cột một lượt
df.loc[df["price"] < 30000, ["neighbourhood", "room_type", "price"]].head()

In [ ]:
# Thêm cột mới bằng công thức vector hoá: quy đổi CLP -> triệu VND (1 CLP ≈ 28 VND)
df["price_trieu_vnd"] = df["price"] * 28 / 1e6
df[["price", "price_trieu_vnd"]].describe().round(2).loc[["50%", "max"]]

## 3. Thống kê để trả lời câu hỏi phân tích

In [ ]:
# Tỷ lệ nguyên căn — mean trên bool trả về tỷ lệ (quy tắc ở bài 3)
(df["room_type"] == "Entire home/apt").mean().round(3)

In [ ]:
# Giới thiệu groupby (bài 5 học kỹ): giá trung vị theo loại phòng
df.groupby("room_type")["price"].median()

In [ ]:
# Các phòng có giá lớn nhất — kiểm tra trực tiếp ứng viên ngoại lai
df.nlargest(3, "price")[["name", "neighbourhood", "price"]]

In [ ]:
# Method chaining: lọc -> nhóm -> trung vị -> 5 giá trị lớn nhất
(df[df["room_type"] == "Entire home/apt"]
   .groupby("neighbourhood")["price"]
   .median()
   .nlargest(5))

⚠️ *La Granja* xuất hiện trong nhóm giá cao nhưng chỉ có 5 phòng. Trung vị của một nhóm nhỏ
cần được diễn giải thận trọng. **Con số tổng hợp luôn cần kèm cỡ nhóm**.

In [ ]:
(df[df["room_type"] == "Entire home/apt"]
   .groupby("neighbourhood")["price"]
   .agg(["median", "size"])          # thêm cỡ nhóm bên cạnh trung vị
   .nlargest(5, "median"))

## 4. Giá trị thiếu — đo, chưa xử

In [ ]:
df.isna().sum().sort_values(ascending=False).head(6)

In [ ]:
# Tỷ lệ thiếu, dạng %
(df.isna().mean().sort_values(ascending=False).head(6) * 100).round(1)

`neighbourhood_group` trống 100% (cột "chết"), `license` ~99%, `price` ~4,6%.
Bỏ dòng? Điền giá trị? — **quyết định phân tích** của bài 10, bài này chỉ cần đo được.

## 5. Ghi kết quả

In [ ]:
bang_gia = df.groupby("room_type")["price"].agg(["median", "mean", "count"]).round(0)
bang_gia.to_csv("gia_theo_loai.csv")
print(open("gia_theo_loai.csv").read())

## 6. Bài tập tại lớp

### Bài 1 — Thói quen 5 bước, tự chạy

Cột `minimum_nights` (số đêm tối thiểu) chưa được đụng đến. Hãy: `describe()` nó, tìm giá trị
lớn nhất, và đếm số phòng đòi ở **trên 365 đêm**. Con số đó nói lên điều gì?

In [ ]:
# TODO Bài 1:
print(df["minimum_nights"].describe().round(1))
print("Số phòng > 365 đêm:", (df["minimum_nights"] > 365).sum())

### Bài 2 — Lọc kết hợp

Đếm số phòng thoả mãn **cả ba** điều kiện: ở Providencia, là `Private room`, giá dưới 40.000 CLP.
Sau đó tính giá trung vị của nhóm này.

In [ ]:
# TODO Bài 2:
nhom = df[(df["neighbourhood"] == "Providencia")
          & (df["room_type"] == "Private room")
          & (df["price"] < 40000)]
print(len(nhom), "phòng | trung vị:", nhom["price"].median())

### Bài 3 — Một câu hỏi, một chuỗi thao tác

Viết **một** chuỗi (method chaining) trả lời: *"5 quận có nhiều listing nhất, mỗi quận bao nhiêu
phòng và tỷ lệ phòng có review trong 12 tháng qua (`number_of_reviews_ltm > 0`) là bao nhiêu?"*

Gợi ý: tạo cột bool trước, rồi `groupby("neighbourhood").agg(...)`, rồi `nlargest`.

In [ ]:
# TODO Bài 3:
df["co_review_ltm"] = df["number_of_reviews_ltm"] > 0
(df.groupby("neighbourhood")
   .agg(so_phong=("id", "size"), ty_le_review=("co_review_ltm", "mean"))
   .nlargest(5, "so_phong")
   .round(3))

## 7. Bài tập về nhà

Chọn **một thành phố khác** (ví dụ Rio de Janeiro:
`https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/2026-06-24/visualisations/listings.csv`)
và viết một đoạn so sánh khoảng 10 dòng với Santiago:

1. Thành phố nào nhiều listing hơn? Tỷ lệ nguyên căn khác nhau thế nào?
2. Giá trung vị theo loại phòng (chú ý: **đơn vị tiền tệ hai nước khác nhau** — so sánh thế nào
   hợp lý? Thay vì so số tuyệt đối, có thể so *cấu trúc*, chẳng hạn tỷ lệ giá Private/Entire).
3. Mức thiếu dữ liệu cột `price` — thành phố nào có tỷ lệ thiếu thấp hơn?

Trình bày mỗi kết luận kèm theo một dòng code làm bằng chứng.

In [ ]:
RUN_CHALLENGE = False   # đổi True rồi làm

if RUN_CHALLENGE:
    URL_RIO = ("https://data.insideairbnb.com/brazil/rj/rio-de-janeiro/"
               "2026-06-24/visualisations/listings.csv")
    rio = pd.read_csv(URL_RIO)
    ...

---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Thói quen 5 bước: shape → sample → info → describe → value_counts | Kiểm tra dataset trước khi phân tích hoặc dùng code do AI tạo |
| Lọc bool, `isin`, `loc[hàng, cột]` | Các thao tác chọn và lọc dữ liệu thường dùng |
| mean trên bool = tỷ lệ; median cho phân phối lệch | Cơ sở để tính và diễn giải KPI |
| Con số tổng hợp phải kèm cỡ nhóm | Tránh kết luận từ nhóm có quá ít quan sát |
| `isna().sum()` đo mức thiếu | Đầu vào cho bài 10 |

**Bài sau:** học kỹ hơn — index, `loc/iloc` khi index không còn là 0,1,2…, `map/apply`,
và `groupby`.